RUNTIME: CPU (no GPU needed for this notebook)    
INSTRUCTIONS:

1. Set runtime to CPU: Runtime → Change runtime type → None
2. Run all cells in order: Runtime → Run all
3. When prompted, authorize Google Drive access
4. Model checkpoints are saved to Google Drive via:
```
    import joblib
    joblib.dump(model, SAVE_DIR + "checkpoints/svm_model.joblib")
```
Make sure SAVE_DIR is set correctly in the setup cell    

5. To save results/figures to GitHub:    
    a. Right-click the file in the left sidebar → Download    
    b. Go to the GitHub repo → results/ → Add file → Upload files    
    c. Commit directly on GitHub
6. Save the notebook: Ctrl+S → save to GitHub when prompted

In [30]:
# ── 1. Mount Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/cs4120_project/" #change if desired

# ── 2. Standard imports ───────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
# ── 3. Data Loading & Preprocessing ───────────────────────────────
dataset = load_dataset("google-research-datasets/go_emotions", "simplified")

train_df_full = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

train_df_full["labels"] = train_df_full["labels"].apply(lambda x: list(map(int, x)))
test_df["labels"] = test_df["labels"].apply(lambda x: list(map(int, x)))

os.makedirs(SAVE_DIR + "data", exist_ok=True)

test_df.to_csv(SAVE_DIR + "data/test.csv", index=False)
print(f"Saved test.csv to {SAVE_DIR}data/test.csv")

train_df_full.to_csv(SAVE_DIR + "data/train.csv", index=False)
print(f"Saved train.csv to {SAVE_DIR}data/train.csv")

fractional_data_files = {
    0.01: "data/train_1pct.csv",
    0.05: "data/train_5pct.csv",
    0.10: "data/train_10pct.csv",
    0.25: "data/train_25pct.csv",
    0.50: "data/train_50pct.csv",
}

for frac, filename in fractional_data_files.items():
    sampled_df = train_df_full.sample(frac=frac, random_state=42)
    sampled_df.to_csv(SAVE_DIR + filename, index=False)
    print(f"Saved {filename} ({frac*100}%) to {SAVE_DIR}{filename}")

print("All data files generated and saved to Google Drive!")

Saved test.csv to /content/drive/MyDrive/cs4120_project/data/test.csv
Saved train.csv to /content/drive/MyDrive/cs4120_project/data/train.csv
Saved data/train_1pct.csv (1.0%) to /content/drive/MyDrive/cs4120_project/data/train_1pct.csv
Saved data/train_5pct.csv (5.0%) to /content/drive/MyDrive/cs4120_project/data/train_5pct.csv
Saved data/train_10pct.csv (10.0%) to /content/drive/MyDrive/cs4120_project/data/train_10pct.csv
Saved data/train_25pct.csv (25.0%) to /content/drive/MyDrive/cs4120_project/data/train_25pct.csv
Saved data/train_50pct.csv (50.0%) to /content/drive/MyDrive/cs4120_project/data/train_50pct.csv
All data files generated and saved to Google Drive!


In [32]:
# ── 3. Data Loading & Preprocessing ───────────────────────────────
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
import joblib
import os

os.makedirs("results", exist_ok=True)
os.makedirs(SAVE_DIR + "data", exist_ok=True)

data_files = {
    0.01: "data/train_1pct.csv",
    0.05: "data/train_5pct.csv",
    0.10: "data/train_10pct.csv",
    0.25: "data/train_25pct.csv",
    0.50: "data/train_50pct.csv",
    1.00: "data/train.csv"
}

test_data_path = SAVE_DIR + "data/test.csv"

if not os.path.exists(test_data_path):
    print(f"Error: Test data file not found at {test_data_path}.")
    print("Please ensure the data generation cell (e.g., cell 49239151) has been run successfully and completed.")
    raise FileNotFoundError(f"Test data file not found: {test_data_path}")

test_df = pd.read_csv(test_data_path)

test_df["labels"] = test_df["labels"].apply(ast.literal_eval)

dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
label_names = dataset["train"].features["labels"].feature.names

mlb = MultiLabelBinarizer(classes=list(range(len(label_names))))
mlb.fit([list(range(len(label_names)))])

y_test = mlb.transform(test_df["labels"])
text_col = "text_clean_tfidf"
if text_col not in test_df.columns:
    text_col = "text"

In [33]:
# ── 4. Training and Evaluation Loop ───────────────────────────────
results = []
SEED = 42
svm_models = {}

for frac, filepath in data_files.items():
    print(f"Training on {frac * 100}% of data ({filepath})...")


    try:
        train_df = pd.read_csv(SAVE_DIR + filepath)
    except FileNotFoundError:
        print(f"File {SAVE_DIR + filepath} not found. Skipping...")
        continue

    train_df["labels"] = train_df["labels"].apply(ast.literal_eval)
    y_train = mlb.transform(train_df["labels"])

    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

    train_text = train_df[text_col].fillna("")
    test_text = test_df[text_col].fillna("")

    X_train = vectorizer.fit_transform(train_text)
    X_test = vectorizer.transform(test_text)

    svm = OneVsRestClassifier(LinearSVC(random_state=SEED, class_weight='balanced'))
    svm.fit(X_train, y_train)

    if frac == 1.00:
        os.makedirs(SAVE_DIR + "checkpoints", exist_ok=True)
        joblib.dump((vectorizer, svm), SAVE_DIR + "checkpoints/svm_model.joblib")
        print("Saved full model to Google Drive!")

    y_pred = svm.predict(X_test)
    report = classification_report(y_test, y_pred, target_names=label_names, output_dict=True, zero_division=0)

    for emotion in label_names:
        metrics = report.get(emotion, {"f1-score": 0, "precision": 0, "recall": 0})
        results.append({
            "method": "svm_tfidf",
            "data_fraction": frac,
            "seed": SEED,
            "emotion": emotion,
            "f1": metrics["f1-score"],
            "precision": metrics["precision"],
            "recall": metrics["recall"]
        })

print("Experimentation complete!")

Training on 1.0% of data (data/train_1pct.csv)...
Training on 5.0% of data (data/train_5pct.csv)...
Training on 10.0% of data (data/train_10pct.csv)...
Training on 25.0% of data (data/train_25pct.csv)...
Training on 50.0% of data (data/train_50pct.csv)...
Training on 100.0% of data (data/train.csv)...
Saved full model to Google Drive!
Experimentation complete!


In [34]:
# ── 5. Save Results ───────────────────────────────────────────────
import pandas as pd

if results:
    results_df = pd.DataFrame(results)
    display(results_df.head(10))

    results_path = "results/svm_tfidf_results.csv"
    results_df.to_csv(results_path, index=False)
    print(f"Results saved to {results_path}.")
else:
    print("No results to save.")

,method,data_fraction,seed,emotion,f1,precision,recall
0,svm_tfidf,0.01,42,admiration,0.170543,0.390071,0.109127
1,svm_tfidf,0.01,42,amusement,0.370588,0.828947,0.238636
2,svm_tfidf,0.01,42,anger,0.063636,0.318182,0.035354
3,svm_tfidf,0.01,42,annoyance,0.023529,0.200000,0.012500
4,svm_tfidf,0.01,42,approval,0.016260,0.166667,0.008547
5,svm_tfidf,0.01,42,caring,0.049383,0.148148,0.029630
6,svm_tfidf,0.01,42,confusion,0.012821,0.333333,0.006536
7,svm_tfidf,0.01,42,curiosity,0.039735,0.333333,0.021127
8,svm_tfidf,0.01,42,desire,0.023810,1.000000,0.012048
9,svm_tfidf,0.01,42,disappointment,0.000000,0.000000,0.000000


Results saved to results/svm_tfidf_results.csv.
